In [7]:
# ============================================================
# TASK 04 — Road Accident Data Analysis
# Patterns: Road Conditions | Weather | Time of Day
# Dataset : Road_Accident_Data.csv  (~308K records)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#3a3d5c',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#b0b0b0',
    'ytick.color':      '#b0b0b0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2a2d3e',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
})

PALETTE = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db',
           '#9b59b6', '#1abc9c', '#e67e22', '#e91e63']

# ── 1. Load & clean ───────────────────────────────────────────
print("Loading dataset …")
df = pd.read_csv('Road Accident Data.csv')
print(f"  Rows: {len(df):,}  |  Columns: {df.shape[1]}")

# Parse date & time
df['Accident Date'] = pd.to_datetime(df['Accident Date'], dayfirst=True, errors='coerce')
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.hour
df['Month'] = df['Accident Date'].dt.month
df['Year']  = df['Accident Date'].dt.year

# Normalise severity (typo 'Fetal' → 'Fatal')
df['Accident_Severity'] = df['Accident_Severity'].replace('Fetal', 'Fatal')

# Hour buckets
def time_bucket(h):
    if pd.isna(h):    return 'Unknown'
    if 0  <= h < 6:   return 'Night (0-6)'
    if 6  <= h < 12:  return 'Morning (6-12)'
    if 12 <= h < 18:  return 'Afternoon (12-18)'
    return 'Evening (18-24)'

df['Time_Period'] = df['Hour'].apply(time_bucket)

SEV_ORDER = ['Slight', 'Serious', 'Fatal']
SEV_COLORS = {'Slight': '#3498db', 'Serious': '#f39c12', 'Fatal': '#e74c3c'}

# ============================================================
# FIGURE 1 — Overview KPI banner
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('ROAD ACCIDENT DATA — OVERVIEW', fontsize=18,
             fontweight='bold', color='white', y=1.02)

kpis = [
    ('Total\nAccidents',   f"{len(df):,}",           '#3498db'),
    ('Total\nCasualties',  f"{df['Number_of_Casualties'].sum():,}", '#e74c3c'),
    ('Fatal\nAccidents',   f"{(df['Accident_Severity']=='Fatal').sum():,}", '#e74c3c'),
    ('Avg Vehicles\n/Accident', f"{df['Number_of_Vehicles'].mean():.2f}", '#2ecc71'),
]
for ax, (label, val, col) in zip(axes, kpis):
    ax.set_facecolor('#1a1d2e')
    ax.text(0.5, 0.6, val,  ha='center', va='center', fontsize=30,
            fontweight='bold', color=col, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=13,
            color='#b0b0b0', transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.savefig('fig1_kpi_overview.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig1_kpi_overview.png")

# ============================================================
# FIGURE 2 — Time-of-Day Analysis  (2 charts)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('TIME OF DAY PATTERNS', fontsize=16, fontweight='bold',
             color='white', y=1.01)

# 2a — Hourly accident count
hourly = df.groupby('Hour').size().reset_index(name='Count')
ax = axes[0]
bars = ax.bar(hourly['Hour'], hourly['Count'],
              color=[PALETTE[2] if (7 <= h <= 9 or 16 <= h <= 19)
                     else PALETTE[0] for h in hourly['Hour']],
              edgecolor='none', width=0.8)
ax.set_title('Accidents by Hour of Day', fontsize=13, color='white', pad=10)
ax.set_xlabel('Hour')
ax.set_ylabel('Number of Accidents')
ax.set_xticks(range(0, 24))
ax.grid(axis='y')
# Annotations for peaks
for h in [8, 17]:
    row = hourly[hourly['Hour'] == h]
    if not row.empty:
        ax.annotate(f"Peak\n{row['Count'].values[0]:,}",
                    xy=(h, row['Count'].values[0]),
                    xytext=(h + 1.5, row['Count'].values[0] + 500),
                    color='#f39c12', fontsize=9, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#f39c12', lw=1.5))

peak_patch  = mpatches.Patch(color=PALETTE[2], label='Rush Hours')
other_patch = mpatches.Patch(color=PALETTE[0], label='Other Hours')
ax.legend(handles=[peak_patch, other_patch], loc='upper left')

# 2b — Severity split by time period
tp_sev = (df.groupby(['Time_Period', 'Accident_Severity'])
            .size().unstack(fill_value=0))
# Reorder columns and rows
tp_order = ['Morning (6-12)', 'Afternoon (12-18)', 'Evening (18-24)', 'Night (0-6)']
tp_sev = tp_sev.reindex(tp_order)
tp_sev = tp_sev[[s for s in SEV_ORDER if s in tp_sev.columns]]

ax2 = axes[1]
bottom = np.zeros(len(tp_sev))
for sev in tp_sev.columns:
    vals = tp_sev[sev].values
    ax2.bar(tp_sev.index, vals, bottom=bottom,
            color=SEV_COLORS[sev], label=sev, edgecolor='none')
    bottom += vals

ax2.set_title('Accident Severity by Time Period', fontsize=13, color='white', pad=10)
ax2.set_xlabel('Time Period')
ax2.set_ylabel('Number of Accidents')
ax2.legend(loc='upper right')
ax2.grid(axis='y')

plt.tight_layout()
plt.savefig('fig2_time_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig2_time_analysis.png")

# ============================================================
# FIGURE 3 — Weather Conditions Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('WEATHER CONDITIONS ANALYSIS', fontsize=16, fontweight='bold',
             color='white', y=1.01)

# 3a — Accident count by weather
weather_counts = df['Weather_Conditions'].value_counts().head(8)
ax = axes[0]
bars = ax.barh(weather_counts.index[::-1], weather_counts.values[::-1],
               color=PALETTE[:len(weather_counts)], edgecolor='none')
ax.set_title('Accidents by Weather Condition', fontsize=13, color='white', pad=10)
ax.set_xlabel('Number of Accidents')
ax.grid(axis='x')
for bar, val in zip(bars, weather_counts.values[::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color='#e0e0e0')

# 3b — Fatal rate by weather
weather_fatal = df.groupby('Weather_Conditions').apply(
    lambda x: (x['Accident_Severity'] == 'Fatal').sum() / len(x) * 100
).sort_values(ascending=False).head(8)

ax2 = axes[1]
colors_w = ['#e74c3c' if r > 2 else '#f39c12' if r > 1 else '#3498db'
            for r in weather_fatal.values]
bars2 = ax2.barh(weather_fatal.index[::-1], weather_fatal.values[::-1],
                 color=colors_w[::-1], edgecolor='none')
ax2.set_title('Fatality Rate (%) by Weather', fontsize=13, color='white', pad=10)
ax2.set_xlabel('Fatality Rate (%)')
ax2.grid(axis='x')
for bar, val in zip(bars2, weather_fatal.values[::-1]):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}%', va='center', fontsize=9, color='#e0e0e0')

plt.tight_layout()
plt.savefig('fig3_weather_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig3_weather_analysis.png")

# ============================================================
# FIGURE 4 — Road Conditions Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('ROAD CONDITIONS ANALYSIS', fontsize=16, fontweight='bold',
             color='white', y=1.01)

# 4a — Pie: road surface
surface_counts = df['Road_Surface_Conditions'].value_counts()
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    surface_counts.values, labels=surface_counts.index,
    autopct='%1.1f%%', colors=PALETTE[:len(surface_counts)],
    startangle=140, pctdistance=0.82,
    wedgeprops=dict(edgecolor='#0f1117', linewidth=2))
for t in texts:     t.set_color('#e0e0e0')
for t in autotexts: t.set_color('white'); t.set_fontsize(9)
ax.set_title('Accidents by Road Surface', fontsize=13, color='white', pad=10)

# 4b — Fatal rate by road surface
surf_fatal = df.groupby('Road_Surface_Conditions').apply(
    lambda x: (x['Accident_Severity'] == 'Fatal').sum() / len(x) * 100
).sort_values(ascending=False)

ax2 = axes[1]
colors_s = ['#e74c3c' if r > 2 else '#f39c12' if r > 1 else '#3498db'
            for r in surf_fatal.values]
ax2.bar(surf_fatal.index, surf_fatal.values, color=colors_s, edgecolor='none')
ax2.set_title('Fatality Rate (%) by Road Surface', fontsize=13, color='white', pad=10)
ax2.set_xlabel('Road Surface')
ax2.set_ylabel('Fatality Rate (%)')
ax2.tick_params(axis='x', rotation=25)
ax2.grid(axis='y')
for i, (label, val) in enumerate(surf_fatal.items()):
    ax2.text(i, val + 0.05, f'{val:.2f}%', ha='center', fontsize=9, color='#e0e0e0')

plt.tight_layout()
plt.savefig('fig4_road_conditions.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig4_road_conditions.png")

# ============================================================
# FIGURE 5 — Day-of-Week & Light Conditions Heatmap
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('TEMPORAL & LIGHT PATTERNS', fontsize=16, fontweight='bold',
             color='white', y=1.01)

# 5a — Day of week bar
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df['Day_of_Week'].value_counts().reindex(day_order)
ax = axes[0]
colors_d = ['#e74c3c' if d in ['Friday','Saturday'] else '#3498db' for d in day_order]
ax.bar(day_counts.index, day_counts.values, color=colors_d, edgecolor='none')
ax.set_title('Accidents by Day of Week', fontsize=13, color='white', pad=10)
ax.set_ylabel('Number of Accidents')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y')

# 5b — Severity by light condition
light_sev = (df.groupby(['Light_Conditions', 'Accident_Severity'])
               .size().unstack(fill_value=0))
light_sev = light_sev[[s for s in SEV_ORDER if s in light_sev.columns]]

ax2 = axes[1]
x = np.arange(len(light_sev))
width = 0.25
for i, sev in enumerate(light_sev.columns):
    ax2.bar(x + i*width, light_sev[sev].values,
            width=width, color=SEV_COLORS[sev], label=sev, edgecolor='none')
ax2.set_title('Severity by Light Condition', fontsize=13, color='white', pad=10)
ax2.set_xticks(x + width)
ax2.set_xticklabels(light_sev.index, rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('Number of Accidents')
ax2.legend()
ax2.grid(axis='y')

plt.tight_layout()
plt.savefig('fig5_day_light.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig5_day_light.png")

# ============================================================
# FIGURE 6 — Road Type & Speed Limit Heatmap
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('ROAD TYPE & SPEED ANALYSIS', fontsize=16, fontweight='bold',
             color='white', y=1.01)

# 6a — accidents by road type
rt_counts = df['Road_Type'].value_counts()
ax = axes[0]
ax.bar(rt_counts.index, rt_counts.values,
       color=PALETTE[:len(rt_counts)], edgecolor='none')
ax.set_title('Accidents by Road Type', fontsize=13, color='white', pad=10)
ax.set_ylabel('Number of Accidents')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y')

# 6b — heatmap: speed limit × severity (% fatal)
speed_sev = (df.groupby(['Speed_limit', 'Accident_Severity'])
               .size().unstack(fill_value=0))
speed_sev = speed_sev[[s for s in SEV_ORDER if s in speed_sev.columns]]
speed_sev_pct = speed_sev.div(speed_sev.sum(axis=1), axis=0) * 100

ax2 = axes[1]
sns.heatmap(speed_sev_pct.T, ax=ax2, annot=True, fmt='.1f',
            cmap='RdYlGn_r', linewidths=0.5,
            cbar_kws={'label': '% of accidents'})
ax2.set_title('Severity % by Speed Limit', fontsize=13, color='white', pad=10)
ax2.set_xlabel('Speed Limit (mph)')
ax2.set_ylabel('Severity')

plt.tight_layout()
plt.savefig('fig6_road_speed.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig6_road_speed.png")

# ============================================================
# FIGURE 7 — Contributing Factors Summary (Multi-factor)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('CONTRIBUTING FACTORS — DEEP DIVE', fontsize=16,
             fontweight='bold', color='white', y=1.01)

# 7a — Casualty heatmap: Hour × Day
pivot = df.pivot_table(values='Number_of_Casualties',
                       index='Day_of_Week', columns='Hour',
                       aggfunc='sum')
pivot = pivot.reindex(day_order)
sns.heatmap(pivot, ax=axes[0,0], cmap='YlOrRd', linewidths=0,
            cbar_kws={'label': 'Casualties'})
axes[0,0].set_title('Casualties Heatmap: Day × Hour', fontsize=12, color='white')
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Day of Week')

# 7b — Urban vs Rural severity
ur_sev = (df.groupby(['Urban_or_Rural_Area', 'Accident_Severity'])
            .size().unstack(fill_value=0))
ur_sev = ur_sev[[s for s in SEV_ORDER if s in ur_sev.columns]]
ur_sev_pct = ur_sev.div(ur_sev.sum(axis=1), axis=0) * 100
ur_sev_pct.plot(kind='bar', ax=axes[0,1],
                color=[SEV_COLORS[s] for s in ur_sev_pct.columns],
                edgecolor='none')
axes[0,1].set_title('Severity Split: Urban vs Rural', fontsize=12, color='white')
axes[0,1].set_ylabel('% of Accidents')
axes[0,1].tick_params(axis='x', rotation=0)
axes[0,1].legend(loc='upper right')
axes[0,1].grid(axis='y')

# 7c — Top vehicle types by fatality count
veh_fatal = df[df['Accident_Severity']=='Fatal']['Vehicle_Type'].value_counts().head(8)
axes[1,0].barh(veh_fatal.index[::-1], veh_fatal.values[::-1],
               color=PALETTE[:len(veh_fatal)], edgecolor='none')
axes[1,0].set_title('Fatal Accidents by Vehicle Type', fontsize=12, color='white')
axes[1,0].set_xlabel('Fatal Accidents')
axes[1,0].grid(axis='x')

# 7d — Monthly trend
monthly = df.groupby(['Month']).size()
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
axes[1,1].plot(monthly.index, monthly.values,
               color='#3498db', linewidth=2.5, marker='o',
               markersize=7, markerfacecolor='#e74c3c')
axes[1,1].fill_between(monthly.index, monthly.values,
                        alpha=0.2, color='#3498db')
axes[1,1].set_xticks(range(1,13))
axes[1,1].set_xticklabels(month_names, rotation=30)
axes[1,1].set_title('Monthly Accident Trend', fontsize=12, color='white')
axes[1,1].set_ylabel('Number of Accidents')
axes[1,1].grid(axis='y')

plt.tight_layout()
plt.savefig('fig7_contributing_factors.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig7_contributing_factors.png")

# ============================================================
# FIGURE 8 — Geographic Hotspot scatter
# ============================================================
print("Generating hotspot map …")
sample = df[['Latitude','Longitude','Accident_Severity']].dropna().sample(
    n=min(30000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')

for sev, grp in sample.groupby('Accident_Severity'):
    ax.scatter(grp['Longitude'], grp['Latitude'],
               c=SEV_COLORS.get(sev, '#888888'), s=2,
               alpha=0.35, label=sev, rasterized=True)

ax.set_title('Accident Hotspot Map (Geographic Distribution)',
             fontsize=15, fontweight='bold', color='white', pad=12)
ax.set_xlabel('Longitude', color='white')
ax.set_ylabel('Latitude',  color='white')
ax.legend(markerscale=5, loc='upper left',
          framealpha=0.3, facecolor='#1a1d2e')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('fig8_hotspot_map.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.close()
print("Saved fig8_hotspot_map.png")

# ============================================================
# Print Key Insights
# ============================================================
print("\n" + "="*60)
print("         KEY INSIGHTS")
print("="*60)
peak_hour = df['Hour'].value_counts().idxmax()
print(f"  Peak accident hour       : {peak_hour}:00")
print(f"  Most dangerous weather   : {df[df['Accident_Severity']=='Fatal']['Weather_Conditions'].value_counts().idxmax()}")
print(f"  Most dangerous surface   : {df[df['Accident_Severity']=='Fatal']['Road_Surface_Conditions'].value_counts().idxmax()}")
print(f"  Highest fatality day     : {df[df['Accident_Severity']=='Fatal']['Day_of_Week'].value_counts().idxmax()}")
print(f"  Urban fatal accidents    : {(df[df['Accident_Severity']=='Fatal']['Urban_or_Rural_Area']=='Urban').sum():,}")
print(f"  Rural fatal accidents    : {(df[df['Accident_Severity']=='Fatal']['Urban_or_Rural_Area']=='Rural').sum():,}")
print("="*60)
print("\nAll figures saved. Task 04 complete ✓")

Loading dataset …
  Rows: 307,973  |  Columns: 21
Saved fig1_kpi_overview.png
Saved fig2_time_analysis.png
Saved fig3_weather_analysis.png
Saved fig4_road_conditions.png
Saved fig5_day_light.png
Saved fig6_road_speed.png
Saved fig7_contributing_factors.png
Generating hotspot map …
Saved fig8_hotspot_map.png

         KEY INSIGHTS
  Peak accident hour       : 17.0:00
  Most dangerous weather   : Fine no high winds
  Most dangerous surface   : Dry
  Highest fatality day     : Saturday
  Urban fatal accidents    : 1,604
  Rural fatal accidents    : 2,349

All figures saved. Task 04 complete ✓
